### **Installs and Imports**

In [ ]:
!pip install -q transformers datasets peft trl

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

import torch, random
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 62.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### **Load the base model** and tokenizer, put the model on the GPU.

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M"     # BASE, not -Instruct

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Loaded {model_id} on {device} — {sum(p.numel() for p in model.parameters()):,} params")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM-135M on cuda — 134,515,008 params


### **Wrap the Model with LoRA Adapters:**

In [ ]:
lora_config = LoraConfig(
    task_type      = "CAUSAL_LM",                # tells peft this is a next-token LM
    r              = 8,                          # the rank — size of A and B
    lora_alpha     = 16,                         # scaling: effective ΔW = (alpha/r)·B·A
    target_modules = ["q_proj", "v_proj"],       # WHICH layers get adapters
    lora_dropout   = 0.05,                       # dropout on the adapter path
    bias           = "none",                     # don't train bias terms
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **CPT-LoRA** using HuggingFace wikitext-dataset:

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# CPT = raw text, every token counts. Join non-empty lines into one corpus.
train_text = "\n".join(t for t in ds["train"]["text"]      if t.strip())
val_text   = "\n".join(t for t in ds["validation"]["text"] if t.strip())

# Pack into one flat stream — exactly your Shakespeare CPT prep, new source
train_ids = torch.tensor(tokenizer(train_text, add_special_tokens=False)["input_ids"])
val_ids   = torch.tensor(tokenizer(val_text,   add_special_tokens=False)["input_ids"])
print(f"train tokens: {len(train_ids):,} | val tokens: {len(val_ids):,}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

train tokens: 2,543,191 | val tokens: 265,683


### **Hyperparameters + The Batch Loader:**

In [ ]:
block_size, batch_size = 256, 8
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

def get_batch(split):
    d  = train_ids if split == "train" else val_ids
    ix = torch.randint(len(d) - block_size, (batch_size,))
    xb = torch.stack([d[i:i+block_size] for i in ix])
    return xb.to(device)

### **Optimizer** + fixed held-out eval:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        xb = get_batch("val")
        total += model(input_ids=xb, labels=xb).loss.item()
    model.train()
    return total / batches

### **Training loop:**

In [ ]:
model.train()
for step in range(max_steps):
    xb   = get_batch("train")
    loss = model(input_ids=xb, labels=xb).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 3.5165 | val 3.4368
step   50 | train 3.1056 | val 3.3581
step  100 | train 3.1403 | val 3.2515
step  150 | train 3.3519 | val 3.2057
step  200 | train 3.2830 | val 3.0989
step  250 | train 2.9637 | val 3.1925
step  300 | train 3.3700 | val 3.0936
step  350 | train 3.1434 | val 3.1351
step  400 | train 2.7940 | val 3.0616
step  450 | train 3.3434 | val 3.0764
step  499 | train 3.1472 | val 3.0887


### **Generate:**

In [ ]:
def sample(prompt, max_new_tokens=120):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(sample("The history of the Roman Empire"))

The history of the Roman Empire is a fascinating one. The Romans were known for their advances in architecture, engineering and military tactics . It was during this period that the most important monuments , such as the Colosseum ( 1st century AD ) or the Pantheon ( first century AD ), are found on public grounds at Rome . In addition to these impressive structures , it has been noted that there have also been notable archaeological finds throughout its territory including the ruins of the Temple of Juno Maximus . 

 Throughout much of its empire many cities had an official temple dedicated to a god ; a practice which continued into the Byzantine era when


### Before **LoRA-CPT**:

The history of the Roman Empire began in 286 BC, when a large group of people from what is now Turkey were expelled by the Romans. They settled around the city of Rome and became known as the "Romans". The Romans had many different cultures that influenced their way of life - Greek culture was very important to them because it helped shape how they saw themselves today!
One interesting thing about the Romans' relationship with other ancient civilizations like Greece comes up again: there are some similarities between these two groups but also lots more differences too (like language). For example; while Greeks spoke Latin instead of Ancient Greek due its

### After **LoRA-CPT**:

The history of the Roman Empire is a fascinating one. It was not only an empire , it also had its own distinct culture . The Romans were very successful in their conquest and rule over much of Europe until they fell under the power of Germanic tribes at the end of the 5th century AD ; however , as well as some of the smaller empires which arose around this time such as the Byzantine Empire ( which included parts of what are now Turkey, Greece and Bulgaria ) , there has been little historical research on the Romans themselves during this period - despite being thought to have originated from the Roman city of Aquileia in Italy where


### **Save The Model Adapters:**

In [ ]:
model.save_pretrained("smollm-lora-cpt-wikitext")   # saves ONLY the adapters — a few MB, not 500MB



---

## **LoRA-SFT** using Alpaca Datset from HuggingFace:

---



### **Reload and add the adapters:**

In [ ]:
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
model = PeftModel.from_pretrained(base, "smollm-lora-cpt-wikitext", is_trainable=True).to(device)
model.print_trainable_parameters()   # should say ~460,800 trainable — NOT 0

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **Load Dataset** (load + format the SFT data (instruction/response pairs)):

In [ ]:
ds = load_dataset("tatsu-lab/alpaca")          # only a 'train' split exists

def format_pair(row):
    instr, inp, out = row["instruction"], row["input"], row["output"]
    if inp.strip():                            # ~40% of rows carry an 'input'
        prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"
    return prompt, out

all_pairs = [format_pair(r) for r in ds["train"].select(range(3000))]
random.shuffle(all_pairs)
train_pairs, val_pairs = all_pairs[:2700], all_pairs[2700:]   # our own held-out split
print(f"train pairs: {len(train_pairs)} | val pairs: {len(val_pairs)}")

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

train pairs: 2700 | val pairs: 300


### **Hyperparameters:**

In [ ]:
EOS = tokenizer.eos_token_id
PAD = tokenizer.pad_token_id
if PAD is None:                              # SmolLM base tokenizer has no pad token
    tokenizer.pad_token = tokenizer.eos_token
    PAD = tokenizer.eos_token_id             # reuse EOS as the pad id
assert EOS is not None and PAD is not None, (EOS, PAD)
print("EOS:", EOS, "| PAD:", PAD)            # confirm both are real ints

MAX_LEN   = 512
batch_sz  = 4
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

EOS: 0 | PAD: 0


### **Masked-example Builder:**

In [ ]:
def build_example(prompt_text, response_text):
    p = tokenizer(prompt_text,   add_special_tokens=False)["input_ids"]
    r = tokenizer(response_text, add_special_tokens=False)["input_ids"] + [EOS]
    input_ids = (p + r)[:MAX_LEN]
    labels    = ([-100]*len(p) + r)[:MAX_LEN]      # mask prompt → loss only on response
    return input_ids, labels

### **Collate SFT Batches (Packing) and Batch-Loader:**

In [ ]:
def collate(batch_pairs):
    ex = [build_example(p, r) for p, r in batch_pairs]
    maxlen = max(len(ids) for ids, _ in ex)
    input_ids, labels, attn = [], [], []
    for ids, lab in ex:
        pad = maxlen - len(ids)
        input_ids.append(ids + [PAD]  * pad)
        labels.append(   lab + [-100] * pad)       # padding never contributes
        attn.append(     [1]*len(ids) + [0]*pad)   # padding mask
    t = lambda z: torch.tensor(z).to(device)
    return t(input_ids), t(labels), t(attn)

def get_sft_batch(pool):
    return collate(random.sample(pool, batch_sz))

### **Optimizer + Fixed held-out Eval:**

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        input_ids, labels, attn = get_sft_batch(val_pairs)
        total += model(input_ids=input_ids, attention_mask=attn, labels=labels).loss.item()
    model.train()
    return total / batches

### **Training Loop** (SFT: masked labels + attention_mask):

In [ ]:
model.train()
for step in range(max_steps):
    input_ids, labels, attn = get_sft_batch(train_pairs)
    loss = model(input_ids=input_ids, attention_mask=attn, labels=labels).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 1.6326 | val 2.0187
step   50 | train 1.8209 | val 1.7050
step  100 | train 0.7389 | val 1.8164
step  150 | train 2.1804 | val 1.8290
step  200 | train 1.6736 | val 1.6103
step  250 | train 1.4623 | val 1.6417
step  300 | train 1.6832 | val 1.6428
step  350 | train 1.3799 | val 1.8357
step  400 | train 1.6481 | val 1.5948
step  450 | train 1.2567 | val 1.6979
step  499 | train 1.3900 | val 1.6604


### **Generate:**

In [ ]:
def sft_generate(instruction, inp=""):
    prompt = (f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n"
              if inp.strip() else
              f"### Instruction:\n{instruction}\n\n### Response:\n")
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=150, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(sft_generate("Explain photosynthesis in simple terms."))

Photosynthesis is a process that occurs within the chloroplasts of plants, where light energy from sunlight combines with carbon dioxide and water to produce glucose (sugar) as well as oxygen gas . This conversion involves two major steps; photolysis converts H2O into O 2 , while ATP synthase synthesizes NADPH by converting ADP + Pi back into ATP using electrons produced during photophosphorylation..


### **Base Model:**
Photosynthesis occurs when light energy from the sun is absorbed by chlorophyll molecules within plant cells, resulting in a chemical reaction that turns water and carbon dioxide into glucose (a type of sugar) through an electron transfer process called photophosphorylation or chemiosmosis. This unique method allows plants to convert sunlight directly using their own internal capacity for producing ATP—the powerhouse responsible for generating electricity! By doing so efficiently while maintaining stability inside cell walls like those found on leaves; chloroplasts play essential roles during growth cycles involving multiple stages such as germination/ovulation phases followed closely after fertilization until reaching maturity level where reproduction takes place via sexual means including pollination between male and female reproductive organs involved here before releasing fertilized eggs carrying genetic information necessary later forming new individuals


### **After CPT:**
1 . The answer is a true statement

2  In the process of converting sunlight into food through light-dependent reactions , plants use energy from molecules and chemical bonds to convert carbon dioxide ( CO₂ ) present within air or water vapor onto glucose molecule which then can be used as fuel for cellular respiration by living cells

3 . In order to get more oxygen required during combustion, organisms have evolved specialised structures such as leaves that capture solar radiation at night whilst trapping it inside their bodies while allowing animals consuming them access daily without any risk if they were not able to obtain sufficient amounts needed throughout day time when less efficient mechanisms are available like photolysis [ 9 ] / photosynthesis ; these organs enable greater efficiency towards obtaining O₃ gas necessary via chem



### **After SFT:**
Photosynthesis is the process by which plants and other photosynthetic organisms convert light energy into chemical potential from water to produce glucose (sugar) as a source of fuel for growth, development ,and maintenance .

### **Save The Adapters:**

In [ ]:
model.save_pretrained("smollm-lora-sft-alpaca")



---

## **RLHF + DPO (Direct Preference Optimization)**

---



### **Rebuild the SFT'd model and MERGE the SFT adapters into the base:**

`merge_and_unload` computes `W' = W + (alpha/r)·B·A` for every adapted layer and returns a plain model with no LoRA machinery left.

In [ ]:
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
sft   = PeftModel.from_pretrained(base, "smollm-lora-sft-alpaca")   # reattach SFT adapters
model = sft.merge_and_unload()                                      # fold W + (α/r)·B·A → W
# `model` is now an ORDINARY CausalLM whose weights ARE the SFT'd SmolLM

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

### **Load + Format the preference dataset** (standard prompt/chosen/rejected):

In [ ]:
TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"   # SAME template as SFT

raw = load_dataset("Intel/orca_dpo_pairs", split="train")
print(raw[0].keys())   # dict_keys(['system', 'question', 'chosen', 'rejected'])

def to_pref(row):
    return {
        "prompt":   TEMPLATE.format(instruction=row["question"]),
        "chosen":   row["chosen"],
        "rejected": row["rejected"],
    }

pref = raw.map(to_pref, remove_columns=raw.column_names).select(range(2000))
print(pref[0]["prompt"][:120])
print("CHOSEN  :", pref[0]["chosen"][:80])
print("REJECTED:", pref[0]["rejected"][:80])

README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl: reconstructing file:   0%|          |  0.00B / 36.3MB            

orca_rlhf.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

dict_keys(['system', 'question', 'chosen', 'rejected'])


Map:   0%|          | 0/12859 [00:00<?, ? examples/s]

### Instruction:
You will be given a definition of a task first, then some input of the task.
This task is about using t
CHOSEN  : [
  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],
  ["Ajax You
REJECTED:  Sure, I'd be happy to help! Here are the RDF triplets for the input sentence:




### **LoRA config for the DPO stage + The DPO Hyperparameters:**

In [ ]:
peft_config = LoraConfig(
    task_type="CAUSAL_LM", r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none",
)

dpo_config = DPOConfig(
    output_dir                  = "smollm-dpo-orca",
    beta                        = 0.1,        # β — the KL-leash strength from the DPO loss
    learning_rate               = 5e-6,       # tiny, for the reasons from last session
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,          # effective batch = 2 × 4 = 8
    num_train_epochs            = 1,
    max_steps                   = 300,        # cap for a fast Colab run
    max_length                  = 1024,
    warmup_steps                = 0.1,
    lr_scheduler_type           = "cosine",
    logging_steps               = 20,
    bf16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

### **Build the DPOTrainer and Train:**

In [ ]:
trainer = DPOTrainer(
    model           = model,          # the merged SFT model (plain CausalLM)
    ref_model       = None,           # None → reference = this model with the DPO adapter DISABLED
    args            = dpo_config,
    train_dataset   = pref,
    processing_class= tokenizer,      # TRL's current name for the tokenizer argument
    peft_config     = peft_config,    # fresh adapters, added on top of the merged SFT base
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
20,0.702299
40,0.698735
60,0.685094
80,0.692846
100,0.691244
120,0.681882
140,0.680722
160,0.674484
180,0.677651
200,0.663809


TrainOutput(global_step=300, training_loss=0.6787638028462728, metrics={'train_runtime': 1962.2641, 'train_samples_per_second': 1.223, 'train_steps_per_second': 0.153, 'total_flos': 1785088616085504.0, 'train_loss': 0.6787638028462728, 'epoch': 1.2152284263959392})

### **Show Log History:**

In [ ]:
import pandas as pd
logs = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["step","loss","rewards/accuracies","rewards/margins",
                    "rewards/chosen","rewards/rejected"] if c in logs]
print(logs[cols].dropna().to_string(index=False))

 step     loss  rewards/accuracies  rewards/margins  rewards/chosen  rewards/rejected
   20 0.702299            0.425000        -0.015878       -0.002813          0.013065
   40 0.698735            0.493750        -0.008792       -0.003710          0.005082
   60 0.685094            0.550000         0.018272        0.000347         -0.017926
   80 0.692846            0.531250         0.003880       -0.004895         -0.008774
  100 0.691244            0.531250         0.007213       -0.003113         -0.010326
  120 0.681882            0.600000         0.025273        0.003614         -0.021659
  140 0.680722            0.625000         0.027684       -0.011572         -0.039256
  160 0.674484            0.637500         0.041283       -0.008243         -0.049525
  180 0.677651            0.681250         0.034175       -0.014204         -0.048379
  200 0.663809            0.712500         0.062796       -0.010320         -0.073116
  220 0.669544            0.675000         0.049941   

### **Save the DPO adapter:**

In [ ]:
trainer.save_model("smollm-dpo-orca")

### **Generate & Compare:**

In [ ]:
def compare(instruction, max_new_tokens=150, seed=0):
    prompt = TEMPLATE.format(instruction=instruction)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    m = trainer.model; m.eval()
    kw = dict(max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,
              top_p=0.9, repetition_penalty=1.3, pad_token_id=tokenizer.eos_token_id)
    strip = lambda o: tokenizer.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

    torch.manual_seed(seed)                          # same random draw for a fair A/B
    with torch.no_grad(), m.disable_adapter():       # DPO adapter OFF → SFT model
        before = strip(m.generate(**enc, **kw))

    torch.manual_seed(seed)                          # reseed → identical sampling stream
    with torch.no_grad():                            # DPO adapter ON → aligned model
        after = strip(m.generate(**enc, **kw))

    print("PROMPT:", instruction)
    print("\n── SFT (adapter OFF, before DPO) ──\n", before)
    print("\n── DPO (adapter ON, after) ──\n", after)

compare("Explain Photosyntesis.")

PROMPT: Explain Photosyntesis.

── SFT (adapter OFF, before DPO) ──
 Photosynthesis is the process by which plants and other organisms use sunlight to produce glucose, a type of sugar that can be used for energy storage in their cells (i). This conversion from carbon dioxide into water provides them with an abundant source of nutrients needed both inside and outside living things!

── DPO (adapter ON, after) ──
 Photosynthesis is the process by which plants and other organisms use sunlight to produce glucose, a type of sugar that can be used for energy storage in their cells or as fuel when needed . This conversion from water into oxygen produces carbon dioxide gas while releasing nutrients such as nitrogen , phosphorus and potassium necessary for plant growth.


### **PROMPT**: Explain why the sky is blue.

### ── **SFT (adapter OFF, before DPO)** ──
 The reason for the color of skies and sunsets being a beautiful shade that we see during sunset or sunrise, as opposed to other colors such as red because it takes longer time than in winter due to its higher temperature level which makes us perceive more blues from this perspective

### ── **DPO (adapter ON, after)** ──
 The reason for the color of skies being seen as they appear to be clear and bluish due to low atmospheric concentrations, which results in a decrease in red light intensity over time because it takes longer at higher altitudes than lower latitudes such as Antarctica or high mountains where sunlight can reach more directly through clouds that are visible from above them (a phenomenon known as the albedo effect).




---



---

## **Push to HuggingFace-Hub**

---



---



In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Assemble the 3 stage models and push each to the Hub
# Run AFTER all adapters are saved: smollm-lora-cpt-wikitext,
# smollm-lora-sft-alpaca, smollm-dpo-orca all exist on disk.
# ═══════════════════════════════════════════════════════════════════
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

login()  # paste a WRITE token: hf.co/settings/tokens

BASE = "HuggingFaceTB/SmolLM-135M"
USER = "m-lagnajit"   # ← your HF username

def build_and_push(adapter_chain, repo):
    """Merge base→...→last adapter in order, push standalone model."""
    m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
    for adapter_path in adapter_chain:                 # apply in chain order
        m = PeftModel.from_pretrained(m, adapter_path).merge_and_unload()
    m.push_to_hub(repo)
    print(f"pushed → {repo}")

# Stage 2: base + CPT
build_and_push(["smollm-lora-cpt-wikitext"],
               f"{USER}/minigpt-v3-cpt")

# Stage 3: base + CPT + SFT  (SFT was trained against merged-CPT)
build_and_push(["smollm-lora-cpt-wikitext", "smollm-lora-sft-alpaca"],
               f"{USER}/minigpt-v3-sft")

# Stage 4: base + CPT + SFT + DPO  (DPO trained against merged-SFT)
build_and_push(["smollm-lora-cpt-wikitext", "smollm-lora-sft-alpaca",
                "smollm-dpo-orca"],
               f"{USER}/minigpt-v3-dpo")

# tokenizer (same for all stages) → push once to the DPO repo, reuse everywhere
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.push_to_hub(f"{USER}/minigpt-v3-dpo")
print("done — 3 stage models + tokenizer on the Hub")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...1khsnks/model.safetensors:   3%|2         | 16.0MB /  538MB            

pushed → m-lagnajit/minigpt-v3-dpo


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

done — 3 stage models + tokenizer on the Hub
